In [4]:
import json, time
from pathlib import Path
import requests
import pandas as pd

In [5]:
URL = "https://kokkai.ndl.go.jp/api/speech"
RAW = Path("../data/raw/kokkai")
RAW.mkdir(parents=True, exist_ok=True)

def fetch(**params):
    base = {"recordPacking": "json", "maximumRecords": 1}
    r = requests.get(URL, params={**base, **params}, timeout=30)
    print(r.url)
    r.raise_for_status()
    data = r.json()
    if "speechRecord" not in data:
        print(json.dumps(data, ensure_ascii=False, indent=2)[:1000])
    time.sleep(2)
    return data

In [6]:
data = fetch(**{"from": "2025-11-01", "until": "2025-11-30"})
print({k: v for k, v in data.items() if k != "speechRecord"})

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2025-11-01&until=2025-11-30
{'numberOfRecords': 11341, 'numberOfReturn': 1, 'startRecord': 1, 'nextRecordPosition': 2}


In [11]:
rec = data["speechRecord"][0]
for k, v in rec.items():
    print(f"{k:20} {str(v)[:80]!r}")

speechID             '121903968X00220251128_000'
issueID              '121903968X00220251128'
imageKind            '会議録'
searchObject         '0'
session              '219'
nameOfHouse          '衆議院'
nameOfMeeting        '外務委員会'
issue                '第2号'
date                 '2025-11-28'
closing              'None'
speechOrder          '0'
speaker              '会議録情報'
speakerYomi          'None'
speakerGroup         'None'
speakerPosition      'None'
speakerRole          'None'
speech               '令和七年十一月二十八日（金曜日）\r\n\u3000\u3000\u3000\u3000午前九時開議\r\n\u3000出席委員\r\n\u3000\u3000\u3000委員長\u3000國場幸之助君\r\n\u3000\u3000\u3000理事\u3000石橋林太郎君\u3000理事\u3000高木\u3000\u3000啓君\r\n\u3000\u3000\u3000理'
startPage            '1'
speechURL            'https://kokkai.ndl.go.jp/txt/121903968X00220251128/0'
meetingURL           'https://kokkai.ndl.go.jp/txt/121903968X00220251128'
pdfURL               'https://kokkai.ndl.go.jp/img/121903968X00220251128/1'


In [8]:
sample = fetch(
    **{
        "from": "2025-11-01", 
        "until": "2025-11-30", 
        "maximumRecords": 100
    }
)
df = pd.DataFrame(sample["speechRecord"])
df["speakerGroup"].value_counts(dropna=False)

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=100&from=2025-11-01&until=2025-11-30


speakerGroup
自由民主党・無所属の会    49
立憲民主党・無所属      43
NaN             6
自由民主党           2
Name: count, dtype: int64

In [13]:
pd.crosstab(df["speakerGroup"].fillna("(空)"), df["nameOfHouse"])

nameOfHouse,衆議院
speakerGroup,
(空),6
立憲民主党・無所属,43
自由民主党,2
自由民主党・無所属の会,49


In [16]:
cols = ["date", "nameOfMeeting", "speaker", "speakerPosition", "speakerRole", "speakerGroup"]
df[df["speakerGroup"] == "自由民主党"][cols]

,date,nameOfMeeting,speaker,speakerPosition,speakerRole,speakerGroup
15,2025-11-28,外務委員会,永井学,国土交通大臣政務官,None,自由民主党
19,2025-11-28,外務委員会,永井学,国土交通大臣政務官,None,自由民主党


In [ ]:
df[df["speakerGroup"] == "自由民主党・無所属の会"][cols].drop_duplicates("speaker").head()

,date,nameOfMeeting,speaker,speakerPosition,speakerRole,speakerGroup
1,2025-11-28,外務委員会,國場幸之助,NaN,None,自由民主党・無所属の会
4,2025-11-28,外務委員会,石橋林太郎,NaN,None,自由民主党・無所属の会
5,2025-11-28,外務委員会,茂木敏充,外務大臣,None,自由民主党・無所属の会
11,2025-11-28,外務委員会,島田智明,外務大臣政務官,None,自由民主党・無所属の会
74,2025-11-28,外務委員会,井野俊郎,経済産業副大臣・内閣府副大臣,None,自由民主党・無所属の会


In [18]:
d = fetch(**{"speaker": "永井学", "maximumRecords": 100})
pd.DataFrame(d["speechRecord"])[["nameOfHouse", "speakerGroup"]].value_counts()

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=100&speaker=%E6%B0%B8%E4%BA%95%E5%AD%A6


nameOfHouse  speakerGroup
参議院          自由民主党           74
             自由民主党・無所属の会     15
衆議院          自由民主党・無所属の会      7
             自由民主党            4
Name: count, dtype: int64

In [19]:
nd = pd.DataFrame(d["speechRecord"])
nd.groupby(["speakerGroup", "nameOfHouse"])["date"].agg(["min", "max", "count"])

min         max  count
speakerGroup nameOfHouse                               
自由民主党        参議院          2023-11-09  2025-11-20     74
             衆議院          2025-11-19  2025-11-28      4
自由民主党・無所属の会  参議院          2026-03-19  2026-05-28     15
             衆議院          2026-03-06  2026-06-03      7

In [9]:
print(df["speakerPosition"].value_counts(dropna=False).head(10))
print(df["speakerRole"].value_counts(dropna=False))
df.loc[df["speechOrder"] == 0, "speech"].str[:200].tolist()

speakerPosition
NaN                        60
外務大臣                       28
国土交通大臣政務官                   2
厚生労働省大臣官房危機管理・医務技術総括審議官     2
経済産業副大臣・内閣府副大臣              2
内閣官房副長官                     2
外務省大臣官房審議官                  1
外務大臣政務官                     1
厚生労働省大臣官房審議官                1
公正取引委員会委員長                  1
Name: count, dtype: int64
speakerRole
None    100
Name: count, dtype: int64


['令和七年十一月二十八日（金曜日）\r\n\u3000\u3000\u3000\u3000午前九時開議\r\n\u3000出席委員\r\n\u3000\u3000\u3000委員長\u3000國場幸之助君\r\n\u3000\u3000\u3000理事\u3000石橋林太郎君\u3000理事\u3000高木\u3000\u3000啓君\r\n\u3000\u3000\u3000理事\u3000星野\u3000剛士君\u3000理事\u3000亀井亜紀子君\r\n\u3000\u3000\u3000理事\u3000末松\u3000義規君\u3000理事\u3000太\u3000\u3000栄志君\r\n\u3000\u3000\u3000理事\u3000杉本\u3000和巳君\u3000理事\u3000深作ヘスス君\r\n\u3000\u3000\u3000\u3000\u3000\u3000逢沢\u3000一郎君\u3000\u3000\u3000\u3000岩屋\u3000\u3000毅君\r\n\u3000\u3000\u3000\u3000\u3000\u3000英利アルフィヤ君\u3000\u3000\u3000\u3000大空\u3000幸星君\r\n\u3000\u3000']

In [11]:
q = df[df["speakerPosition"].isna() & (df["speechOrder"] > 0)]
print(q["speakerGroup"].value_counts(dropna=False))
q[["speaker", "speakerGroup", "speech"]].assign(speech=q["speech"].str[:40]).head(15)

speakerGroup
立憲民主党・無所属      43
自由民主党・無所属の会    16
Name: count, dtype: int64


,speaker,speakerGroup,speech
1,國場幸之助,自由民主党・無所属の会,○國場委員長 これより会議を開きます。\r\n 国際情勢に関する件について調査を進め
2,國場幸之助,自由民主党・無所属の会,○國場委員長 御異議なしと認めます。よって、そのように決しました。\r\n ―
3,國場幸之助,自由民主党・無所属の会,○國場委員長 質疑の申出がありますので、順次これを許します。石橋林太郎君。
4,石橋林太郎,自由民主党・無所属の会,○石橋委員 皆さん、おはようございます。自民党の石橋林太郎です。\r\n 茂木大臣に
6,石橋林太郎,自由民主党・無所属の会,○石橋委員 御答弁ありがとうございました。\r\n 今御答弁をいただきましたとおり、
8,石橋林太郎,自由民主党・無所属の会,○石橋委員 ありがとうございました。\r\n そのときの事態に対応して、時の政権が命
10,石橋林太郎,自由民主党・無所属の会,○石橋委員 ありがとうございます。\r\n 今、所有権又は管理権というものが、非核兵
12,石橋林太郎,自由民主党・無所属の会,○石橋委員 御答弁ありがとうございました。\r\n 正しい情報を発信することに加えて
13,國場幸之助,自由民主党・無所属の会,○國場委員長 次に、末松義規君。
14,末松義規,立憲民主党・無所属,○末松委員 茂木外務大臣には初めて質問させていただきます。\r\n 私は、今思うんで


In [13]:
for month in ["2026-06", "2026-07", "2026-08", "2026-09"]:
    d = fetch(**{"from": f"{month}-01", "until": f"{month}-28"})
    print(month, d.get("numberOfRecords", 0))

https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-06-01&until=2026-06-28
2026-06 14669
https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-07-01&until=2026-07-28
2026-07 9197
https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-08-01&until=2026-08-28
2026-08 12
https://kokkai.ndl.go.jp/api/speech?recordPacking=json&maximumRecords=1&from=2026-09-01&until=2026-09-28
{
  "numberOfRecords": 0,
  "numberOfReturn": 0,
  "startRecord": 1,
  "nextRecordPosition": null
}
2026-09 0


In [14]:
with open(RAW / "sample_2025-11.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, ensure_ascii=False, indent=1)